# **Reshaping and Pivot Tables**

**DS112 — Computer Programming**

**Week 16:** · **Suggested class time: 5 hours (2-hour lecture + 3-hour laboratory)**

**ILO :** Implement pivot functionalities and reshape data using pandas.

The **same data** can be arranged in different **shapes**:

* **Long** (tidy) format: one row per **observation** — easy to filter, group, and plot.
* **Wide** format: one row per **entity**, with a column per variable or time period — easy for people to read.

```text
LONG                                   WIDE
date        station    temp_c          date        Cagayan de Oro  Iligan  Malaybalay
2026-08-01  CDO        28.1            2026-08-01  28.1            27.9    22.7
2026-08-01  Iligan     27.9            2026-08-02  28.6            28.2    22.4
2026-08-01  Malaybalay 22.7
2026-08-02  CDO        28.6
...
```

> **Analogy.** Reshaping is like rearranging the **same deck of cards**: sorted by suit in rows, or laid out as a grid of suits × ranks. No card is added or lost.

Converting between the two is called **reshaping**, and **pivot tables** combine reshaping with aggregation.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

weather = pd.read_csv("data/weather.csv", parse_dates=["date"])
sales = pd.read_csv("data/canteen_sales.csv", parse_dates=["date"])
menu = pd.read_csv("data/menu.csv")
sales = sales.merge(menu, on="item_code").assign(revenue=lambda d: d["qty"] * d["price"])
weather.head()

## **Long → Wide with `pivot`**

`pivot(index=…, columns=…, values=…)` rearranges data when each index/column pair occurs **once**.

In [ ]:
wide = weather.pivot(index="date", columns="station", values="temp_c")
wide.head()

## **Wide → Long with `melt`**

In [ ]:
long = wide.reset_index().melt(id_vars="date", var_name="station", value_name="temp_c")
print(long.shape, weather.shape[0])
long.head()

In [ ]:
grades = pd.read_csv("data/grades.csv")
scores_long = grades.melt(id_vars=["student_id", "name", "section"],
                          value_vars=["quiz", "lab", "midterm", "final"],
                          var_name="assessment", value_name="score")
scores_long.head(6)

Long format makes questions about **all assessments together** easy:

In [ ]:
scores_long.groupby("assessment")["score"].agg(["mean", "std"]).round(2)

## **Pivot Tables**

When index/column pairs occur **many times**, they must be **aggregated**. `pivot_table` does reshaping **and** aggregation in one step — like the pivot tables in Excel.

In [ ]:
pd.pivot_table(sales, index="item", columns="branch", values="revenue", aggfunc="sum")

In [ ]:
sales["week"] = sales["date"].dt.isocalendar().week
pd.pivot_table(
    sales, index="category", columns="week", values="qty",
    aggfunc="sum", fill_value=0, margins=True, margins_name="Total",
)

In [ ]:
weather["month"] = weather["date"].dt.month_name()
pd.pivot_table(weather, index="station", columns="month",
               values=["temp_c", "rain_mm"], aggfunc={"temp_c": "mean", "rain_mm": "sum"}).round(1)

| Argument | Purpose |
| -------- | ------- |
| `index` | rows of the result |
| `columns` | columns of the result |
| `values` | what to aggregate |
| `aggfunc` | `"sum"`, `"mean"`, `"count"`, a list, or a dict per column |
| `fill_value` | replace empty cells |
| `margins` | add row and column totals |

## **Cross-Tabulation**

`pd.crosstab` counts how often combinations of categories occur:

In [ ]:
grades["remark"] = np.where(grades["final"] >= 75, "Passed", "Failed")
pd.crosstab(grades["section"], grades["remark"], margins=True)

In [ ]:
pd.crosstab(grades["section"], grades["remark"], normalize="index").round(2)     # proportions per row

## **`stack` and `unstack`**

`unstack` moves an index level into the columns (long → wide); `stack` does the reverse.

In [ ]:
by_branch_cat = sales.groupby(["category", "branch"])["revenue"].sum()
table = by_branch_cat.unstack("branch")
print(table)
print(table.stack().head())

## **Choosing a Shape**

| Use **long** format when… | Use **wide** format when… |
| ------------------------- | ------------------------- |
| grouping, filtering, and plotting with pandas | presenting a table to people |
| the number of categories may grow | comparing a few categories side by side |
| storing data (tidy data) | computing differences between columns |

## **Summary**

| Task | Code |
| ---- | ---- |
| long → wide (unique pairs) | `df.pivot(index, columns, values)` |
| wide → long | `df.melt(id_vars, value_vars, var_name, value_name)` |
| reshape + aggregate | `pd.pivot_table(df, index, columns, values, aggfunc, margins=True)` |
| count combinations | `pd.crosstab(a, b, normalize=…)` |
| move index levels | `unstack()`, `stack()` |

## **References**

Maulion, M. R. (2022). *DS 112 computer programming* [Instructional material]. University of Science and Technology of Southern Philippines.

McKinney, W. (2022). *Python for data analysis: Data wrangling with pandas, NumPy, and Jupyter* (3rd ed.). O'Reilly Media. https://wesmckinney.com/book/

The pandas development team. (2026). *Reshaping and pivot tables*. https://pandas.pydata.org/docs/user_guide/reshaping.html

VanderPlas, J. (2023). *Python data science handbook: Essential tools for working with data* (2nd ed.). O'Reilly Media. https://jakevdp.github.io/PythonDataScienceHandbook/

Wickham, H. (2014). Tidy data. *Journal of Statistical Software, 59*(10), 1–23. https://doi.org/10.18637/jss.v059.i10